# Supervisor Architecture & Parallel Agents

Topics:
- **Supervisor pattern**: one coordinator routes work to specialist agents and loops back
- `RouteDecision` — structured routing with `with_structured_output` so the supervisor always returns a valid agent name
- The **supervisor loop**: `supervisor → specialist → supervisor → … → finalize`
- **Parallel fan-out / fan-in**: multiple agents triggered from `START`, converging at a synthesizer
- **Map-Reduce**: batch documents → parallel summarize → reduce into one overview

```
Supervisor pattern:
START → supervisor ──► researcher ──┐
           ▲          ► writer    ──┤ → supervisor (loop)
           │          ► critic    ──┘
           └── FINISH ──────────────► finalize → END

Parallel fan-out/fan-in:
                ┌──► research ──┐
START ──────────┼──► creative ──┼──► synthesize → END
                └──► technical ─┘
```

In [ ]:
from typing import Literal
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict, Annotated
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. Supervisor Architecture

The supervisor pattern has one coordinating agent that:
1. Reads the full conversation
2. Decides which specialist should act next (or finishes)
3. Routes, the specialist runs, then control returns to the supervisor

### Key design: `RouteDecision` with `with_structured_output`

```python
class RouteDecision(BaseModel):
    next: Literal["researcher", "writer", "critic", "FINISH"]
    reasoning: str

supervisor_llm = llm.with_structured_output(RouteDecision)
```

`with_structured_output` forces the LLM to return a valid `next` value from the `Literal` — the graph can use this directly as a routing key without any string parsing.

In [ ]:
class SupervisorState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    next_agent: str
    task_complete: bool
    final_response: str


def create_supervisor_system():
    """Create a supervisor with researcher, writer, and critic specialists."""

    class RouteDecision(BaseModel):
        next: Literal["researcher", "writer", "critic", "FINISH"] = Field(
            description="The next agent to call, or FINISH if task is complete"
        )
        reasoning: str = Field(description="Why this agent was chosen")

    supervisor_llm = llm.with_structured_output(RouteDecision)

    # ------------------------------------------------------------------
    # Supervisor node: reads entire conversation, decides who acts next
    # ------------------------------------------------------------------
    def supervisor(state: SupervisorState) -> dict:
        system_prompt = """You are a supervisor managing a team of specialists:

        1. researcher - Gathers information and facts
        2. writer - Creates content and text
        3. critic - Reviews and improves work

        Based on the conversation, decide which agent should act next.
        If the task is complete, respond with FINISH."""

        messages = [SystemMessage(content=system_prompt)] + state["messages"]
        decision = supervisor_llm.invoke(messages)

        if decision.next == "FINISH":
            return {"next_agent": "FINISH", "task_complete": True}

        return {
            "next_agent": decision.next,
            "messages": [AIMessage(content=f"[Supervisor] Routing to {decision.next}: {decision.reasoning}")],
        }

    # ------------------------------------------------------------------
    # Specialist nodes — each reads conversation context and adds findings
    # ------------------------------------------------------------------
    def researcher(state: SupervisorState) -> dict:
        task = next((m.content for m in state["messages"] if isinstance(m, HumanMessage)), "")
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a research specialist. Gather facts relevant to the task. Be thorough but concise."),
            ("human", "Task context:\n{context}\n\nProvide your research findings."),
        ])
        response = llm.invoke(prompt.format_messages(context=task))
        return {"messages": [AIMessage(content=f"[Researcher] {response.content}")]}

    def writer(state: SupervisorState) -> dict:
        context = "\n".join([m.content for m in state["messages"][-5:]])
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a writing specialist. Create clear, engaging content based on the available information."),
            ("human", "Previous work:\n{context}\n\nWrite the content."),
        ])
        response = llm.invoke(prompt.format_messages(context=context))
        return {"messages": [AIMessage(content=f"[Writer] {response.content}")]}

    def critic(state: SupervisorState) -> dict:
        context = "\n".join([m.content for m in state["messages"][-3:]])
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a quality critic. Review the work and provide constructive feedback."),
            ("human", "Work to review:\n{context}\n\nProvide your critique."),
        ])
        response = llm.invoke(prompt.format_messages(context=context))
        return {"messages": [AIMessage(content=f"[Critic] {response.content}")]}

    # ------------------------------------------------------------------
    # Finalize: extracts the last writer output as the final response
    # ------------------------------------------------------------------
    def finalize(state: SupervisorState) -> dict:
        for msg in reversed(state["messages"]):
            if isinstance(msg, AIMessage) and "[Writer]" in msg.content:
                return {"final_response": msg.content.replace("[Writer] ", "")}
        return {"final_response": "Task completed."}

    # ------------------------------------------------------------------
    # Conditional edge: route_to_agent reads next_agent from state
    # ------------------------------------------------------------------
    def route_to_agent(state: SupervisorState) -> str:
        if state.get("task_complete"):
            return "finalize"
        return state["next_agent"]

    g = StateGraph(SupervisorState)
    g.add_node("supervisor", supervisor)
    g.add_node("researcher", researcher)
    g.add_node("writer", writer)
    g.add_node("critic", critic)
    g.add_node("finalize", finalize)

    g.add_edge(START, "supervisor")
    g.add_conditional_edges(
        "supervisor", route_to_agent,
        {"researcher": "researcher", "writer": "writer", "critic": "critic", "finalize": "finalize"},
    )
    # Specialists loop back to supervisor
    for node in ["researcher", "writer", "critic"]:
        g.add_edge(node, "supervisor")
    g.add_edge("finalize", END)

    return g.compile()

In [ ]:
agent = create_supervisor_system()

result = agent.invoke({
    "messages": [HumanMessage(content="Write a short blog post about the benefits of AI in healthcare")],
    "next_agent": "",
    "task_complete": False,
    "final_response": "",
})

print("Supervisor routing decisions:")
for msg in result["messages"]:
    if isinstance(msg, AIMessage) and "[Supervisor]" in msg.content:
        print(f"  → {msg.content}")

print(f"\nFinal Response (truncated):\n{result['final_response'][:400]}")

## 2. Parallel Fan-Out / Fan-In

Multiple edges from `START` to different nodes trigger them in parallel. All results land in the shared state before the synthesizer runs.

```python
graph.add_edge(START, "research")   # ─┐
graph.add_edge(START, "creative")   #  ├─ all triggered at graph start
graph.add_edge(START, "technical")  # ─┘

graph.add_edge("research",  "synthesize")  # ─┐
graph.add_edge("creative",  "synthesize")  #  ├─ synthesize waits for all
graph.add_edge("technical", "synthesize")  # ─┘
```

LangGraph waits for **all fan-out branches** to complete before calling the synthesizer — no extra coordination code needed.

In [ ]:
class ParallelState(TypedDict):
    query: str
    research_result: str
    creative_result: str
    technical_result: str
    final_synthesis: str


parallel_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)


def create_parallel_research():
    """Three agents working in parallel on the same query."""

    def research_agent(state: ParallelState) -> dict:
        response = parallel_llm.invoke([
            SystemMessage(content="You are an academic researcher. Provide factual, well-sourced information."),
            HumanMessage(content=f"Research this topic: {state['query']}"),
        ])
        return {"research_result": response.content}

    def creative_agent(state: ParallelState) -> dict:
        response = parallel_llm.invoke([
            SystemMessage(content="You are a creative thinker. Provide novel perspectives and ideas."),
            HumanMessage(content=f"Give creative insights on: {state['query']}"),
        ])
        return {"creative_result": response.content}

    def technical_agent(state: ParallelState) -> dict:
        response = parallel_llm.invoke([
            SystemMessage(content="You are a technical analyst. Provide practical, implementation-focused insights."),
            HumanMessage(content=f"Analyze technically: {state['query']}"),
        ])
        return {"technical_result": response.content}

    def synthesize(state: ParallelState) -> dict:
        synthesis_prompt = f"""Synthesize these three perspectives into a comprehensive response:

        RESEARCH: {state['research_result']}

        CREATIVE: {state['creative_result']}

        TECHNICAL: {state['technical_result']}

        Create a unified, well-structured response."""

        response = parallel_llm.invoke([
            SystemMessage(content="You are an expert synthesizer. Combine multiple perspectives into coherent insights."),
            HumanMessage(content=synthesis_prompt),
        ])
        return {"final_synthesis": response.content}

    g = StateGraph(ParallelState)
    g.add_node("research", research_agent)
    g.add_node("creative", creative_agent)
    g.add_node("technical", technical_agent)
    g.add_node("synthesize", synthesize)

    # Fan-out: START triggers all three agents
    g.add_edge(START, "research")
    g.add_edge(START, "creative")
    g.add_edge(START, "technical")

    # Fan-in: all three feed synthesize
    g.add_edge("research",  "synthesize")
    g.add_edge("creative",  "synthesize")
    g.add_edge("technical", "synthesize")

    g.add_edge("synthesize", END)
    return g.compile()


parallel_agent = create_parallel_research()

result = parallel_agent.invoke({
    "query": "The future of remote work",
    "research_result": "",
    "creative_result": "",
    "technical_result": "",
    "final_synthesis": "",
})

print("Individual perspectives (truncated to 200 chars each):")
print(f"[Research]  {result['research_result'][:200]}...")
print(f"[Creative]  {result['creative_result'][:200]}...")
print(f"[Technical] {result['technical_result'][:200]}...")
print(f"\n[Synthesized]\n{result['final_synthesis'][:500]}")

## 3. Map-Reduce Pattern

Map-Reduce applies the same operation to each item in a collection (map), then combines the results (reduce). Here we summarize multiple documents then combine the summaries.

| Phase | Node | What it does |
|-------|------|--------------|
| Map   | `map` | Iterates over `documents`, calls LLM once per doc, collects into `summaries` list |
| Reduce | `reduce` | Takes all `summaries`, calls LLM once to produce `final_summary` |

This version uses a sequential map loop inside one node. For truly parallel map, use the **Send API** (covered in the project notebook).

In [ ]:
class MapReduceState(TypedDict):
    documents: list[str]
    summaries: list[str]
    final_summary: str


def create_map_reduce_summarizer():
    """Summarize multiple documents then combine into one overview."""

    def map_summarize(state: MapReduceState) -> dict:
        """Summarize each document sequentially (map phase)."""
        summaries = []
        for doc in state["documents"]:
            response = llm.invoke([
                SystemMessage(content="Summarize this document in 2-3 sentences."),
                HumanMessage(content=doc),
            ])
            summaries.append(response.content)
        return {"summaries": summaries}

    def reduce_combine(state: MapReduceState) -> dict:
        """Combine all summaries into a single coherent overview (reduce phase)."""
        all_summaries = "\n\n".join(
            [f"Summary {i+1}: {s}" for i, s in enumerate(state["summaries"])]
        )
        response = llm.invoke([
            SystemMessage(content="Combine these summaries into one coherent overview."),
            HumanMessage(content=all_summaries),
        ])
        return {"final_summary": response.content}

    g = StateGraph(MapReduceState)
    g.add_node("map", map_summarize)
    g.add_node("reduce", reduce_combine)
    g.add_edge(START, "map")
    g.add_edge("map", "reduce")
    g.add_edge("reduce", END)
    return g.compile()


documents = [
    "Python is a high-level programming language known for its simplicity and readability. It supports multiple programming paradigms and has a vast ecosystem of libraries.",
    "Machine learning is a subset of AI that enables systems to learn from data. Common approaches include supervised, unsupervised, and reinforcement learning.",
    "Cloud computing provides on-demand access to computing resources. Major providers include AWS, Azure, and Google Cloud Platform.",
]

map_reduce = create_map_reduce_summarizer()
result = map_reduce.invoke({"documents": documents, "summaries": [], "final_summary": ""})

print("Individual summaries (map phase):")
for i, summary in enumerate(result["summaries"]):
    print(f"  {i+1}. {summary}")

print(f"\nCombined overview (reduce phase):\n{result['final_summary']}")